# safety_figures

> Figures for Phase 9: honest and team welfare under attack, defences, cost and benefit, monitoring, detection AUC and H3.

Spec: `specs/multiagent/phase9/` (design-low-level section 6).

In [ ]:
#| default_exp safety_figures

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib.pyplot as plt
import numpy as np

KINDS = ('liar', 'freerider', 'saboteur')
CHANS = ('none', 'blackboard', 'messages', 'hierarchical')
DEFS = ('none', 'trust', 'reputation', 'sanctions')


def _curve(tab, fixed, metric):
    "Points (number of bad agents, mean, lo, hi) of a grouped table, matching `fixed` on the leading keys."
    pts = sorted((k[-1], *v[metric]) for k, v in tab.items() if k[:len(fixed)] == fixed or
                 (k[-1] == 0 and k[:len(fixed) - 1] == fixed[:-1]))
    return np.array(pts, float).reshape(-1, 4)


def _err(ax, p, **kw):
    if len(p):
        ax.errorbar(p[:, 0], p[:, 1], yerr=[p[:, 1] - p[:, 2], p[:, 3] - p[:, 1]], marker='o', ms=3, capsize=2, **kw)


def safetyFigures(tabs):
    "Figures from `safety_report.summaries`; each is drawn only when its table has data."
    figs = {}
    att = tabs.get('attack') or {}
    Ns = sorted({int(k[1]) for k in att})
    for name, metric in (('attack', 'Wh'), ('team', 'Wt')):
        if not att:
            continue
        f, ax = plt.subplots(len(Ns), 3, figsize=(12, 3.2 * len(Ns)), squeeze=False)
        for i, N in enumerate(Ns):
            for j, kind in enumerate(KINDS):
                for ch in CHANS:
                    _err(ax[i][j], _curve(att, (ch, N, kind), metric), label=ch)
                ax[i][j].set_title(f'{kind}, N = {N}', fontsize=9); ax[i][j].set_xlabel('bad agents')
            ax[i][0].set_ylabel('honest welfare / agent-s' if metric == 'Wh' else 'team welfare / agent-s')
        ax[0][0].legend(fontsize=7); figs[name] = f
    dfn = tabs.get('defence') or {}
    if dfn:
        chans = sorted({k[0] for k in dfn}); NsD = sorted({int(k[2]) for k in dfn})
        f, ax = plt.subplots(len(chans), len(NsD), figsize=(5 * len(NsD), 3.4 * len(chans)), squeeze=False)
        for i, ch in enumerate(chans):
            for j, N in enumerate(NsD):
                _err(ax[i][j], _curve(att, (ch, N, 'liar'), 'Wh'), label='none', color='k')
                for d in DEFS[1:]:
                    _err(ax[i][j], _curve({(k[0], k[2], k[3], k[4]): v for k, v in dfn.items() if k[1] == d},
                                          (ch, N, 'liar'), 'Wh'), label=d)
                ax[i][j].set_title(f'{ch}, N = {N}: liars', fontsize=9); ax[i][j].set_xlabel('liars')
        ax[0][0].set_ylabel('honest welfare / agent-s'); ax[0][0].legend(fontsize=7); figs['defence'] = f
    cb = tabs.get('costBenefit') or {}
    if cb:
        keys = sorted(cb)
        f, ax = plt.subplots(figsize=(max(6, 0.7 * len(keys)), 3.6))
        x = np.arange(len(keys))
        for off, m in ((-0.2, 'cost'), (0.2, 'benefit')):
            v = np.array([cb[k][m] for k in keys], float)
            ax.bar(x + off, v[:, 0], 0.4, yerr=[v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]], capsize=2, label=m)
        ax.set_xticks(x, [f'{c[:2]}/{d}/{n}' for c, d, n in keys], rotation=60, fontsize=7); ax.axhline(0, color='k', lw=0.6)
        ax.set_ylabel('share of clean welfare'); ax.legend(fontsize=7); figs['cost'] = f
    mon = tabs.get('monitor') or {}
    if mon:
        NsM = sorted({int(k[2]) for k in mon})
        f, ax = plt.subplots(1, len(NsM), figsize=(5.5 * len(NsM), 3.6), squeeze=False)
        for j, N in enumerate(NsM):
            for d in sorted({k[0] for k in mon}):
                for liars, ls in ((False, '--'), (True, '-')):
                    pts = sorted((v['monitorCost'][0], v['Wh'][0]) for k, v in mon.items()
                                 if k[0] == d and int(k[2]) == N and (int(k[3]) > 0) == liars)
                    if pts:
                        ax[0][j].plot(*np.array(pts).T, ls=ls, marker='o', label=f"{d}{' (25% liars)' if liars else ' (clean)'}")
            ax[0][j].set_title(f'blackboard, N = {N}', fontsize=9); ax[0][j].set_xlabel('monitoring cost (share of agent-s)')
        ax[0][0].set_ylabel('honest welfare / agent-s'); ax[0][0].legend(fontsize=6); figs['monitor'] = f
    det = tabs.get('detection') or {}
    if det:
        keys = sorted(det)
        f, ax = plt.subplots(figsize=(max(6, 0.7 * len(keys)), 3.4))
        x = np.arange(len(keys))
        for off, m in ((-0.2, 'aucWhite'), (0.2, 'aucBlack')):
            ax.bar(x + off, [det[k][m][0] for k in keys], 0.4, label=m)
        ax.set_xticks(x, [f'{c[:2]}/{d}/{int(n)}' for c, d, n in keys], rotation=60, fontsize=7)
        ax.axhline(0.5, color='k', lw=0.6, ls=':'); ax.set_ylim(0, 1.05); ax.legend(fontsize=7); figs['detection'] = f
    h3 = tabs.get('h3') or {}
    if h3:
        f, ax = plt.subplots(figsize=(5.5, 3.6))
        for ch in sorted({k[0] for k in h3}):
            p = np.array(sorted((k[1], *v['loss']) for k, v in h3.items() if k[0] == ch), float)
            _err(ax, p, label=ch)
        ax.set_xscale('log', base=2); ax.axhline(0, color='k', lw=0.6); ax.set_xlabel('team size')
        ax.set_ylabel('relative honest-welfare loss, one liar'); ax.legend(fontsize=7); figs['h3'] = f
    return figs

## Tests

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from fastcore.test import test_eq
from explRL.bench_grid import runGrid
from explRL.safety_figures import safetyFigures
from explRL.safety_grid import attackGrid, defenceGrid, monitorGrid, h3Grid, runSafetyCell
from explRL.safety_report import summaries

rows = runGrid(attackGrid('quick') + defenceGrid('quick') + monitorGrid('quick') + h3Grid('quick'), 2, run=runSafetyCell)
figs = safetyFigures(summaries(rows))
test_eq(set(figs), {'attack', 'team', 'defence', 'cost', 'monitor', 'detection', 'h3'})
test_eq(safetyFigures({}), {})
plt.close('all')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()